# Title + Objective (Markdown)

Compare baseline models fairly with same preprocessing + CV

# Model Baselines — EcoType Forest Cover Classification

## Objective
This notebook trains and compares baseline classification models on the selected EcoType dataset.

### What this notebook does
- loads the selected dataset
- splits data into train and test sets
- builds preprocessing + model pipelines
- trains each baseline model
- evaluates each model using accuracy and macro F1
- stores all results in a metrics table
- saves the baseline comparison table to CSV

### Why this notebook exists
The reusable modeling logic lives in `src/`.
This notebook is the interactive experiment layer used to compare baseline models and decide which ones should move forward to tuning.

# Imports + Load Data (Code)

Load X_train, y_train

Load preprocessor setup

In [18]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
import yaml

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate

project_root = Path.cwd().resolve().parents[0]

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.features.selection import load_data, split_features_target
from src.modeling.models import get_baseline_models
from src.modeling.metrics import evaluate_classification
from src.modeling.train import (
    split_feature_types,
    prepare_modeling_dataframe,
    build_model_pipeline,
    save_baseline_results,
)

print("project_root:", project_root)

project_root: F:\DATA SCIENCE\Projects\Forest Cover Type Prediction


In [3]:
config_dir = project_root / "config"
paths_file = config_dir / "paths.yaml"
train_file = config_dir / "train.yaml"

with open(paths_file, "r", encoding="utf-8") as f:
    paths_cfg = yaml.safe_load(f)

with open(train_file, "r", encoding="utf-8") as f:
    train_cfg = yaml.safe_load(f)

print("Loaded configs successfully.")

Loaded configs successfully.


## Notebook position in pipeline

This notebook comes after:
1. setup and planning
2. data understanding
3. cleaning/transformation
4. feature engineering
5. EDA
6. feature selection
7. preprocessing design

This notebook does not create new features.
It compares baseline models on the selected feature set using the reusable preprocessing pipeline.

In [4]:
data_dir = project_root / paths_cfg["data"]["processed_dir"]
reports_dir = project_root / paths_cfg["artifacts"]["reports_dir"]
figures_dir = project_root / paths_cfg["artifacts"]["figures_dir"]

reports_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

selected_file = data_dir / "forest_cover_selected.csv"
baseline_scores_file = reports_dir / "baseline_scores.csv"

target_col = train_cfg["data"]["target_col"]
drop_cols = train_cfg["data"].get("drop_cols", [])
id_cols = train_cfg["data"].get("id_cols", [])

test_size = train_cfg["split"]["test_size"]
random_state = train_cfg["split"]["random_state"]
n_splits = train_cfg["cv"]["n_splits"]
use_stratify = train_cfg["split"]["stratify"]
use_shuffle = train_cfg["split"]["shuffle"]

print("selected_file:", selected_file)
print("baseline_scores_file:", baseline_scores_file)
print("target_col:", target_col)

selected_file: F:\DATA SCIENCE\Projects\Forest Cover Type Prediction\data\processed\forest_cover_selected.csv
baseline_scores_file: F:\DATA SCIENCE\Projects\Forest Cover Type Prediction\reports\baseline_scores.csv
target_col: cover_type


In [5]:
df = load_data(selected_file)

print("Loaded selected dataset")
print("Shape:", df.shape)
df.head()

Loaded selected dataset
Shape: (145890, 21)


,elevation,aspect,vertical_distance_to_hydrology,horizontal_distance_to_roadways,hillshade_9am,hillshade_3pm,horizontal_distance_to_fire_points,wilderness_area,soil_type,aspect_sin,...,hydrology_distance,hillshade_mean,hillshade_std,elevation_slope_interaction,elevation_slope_ratio,road_fire_gap,road_fire_ratio,hydrology_fire_ratio,hydrology_road_ratio,cover_type
0,2596,51,0,510,221,148,6279,1,29,0.777146,...,258.000000,200.333333,45.654500,7788,865.333045,5769,0.081223,0.041089,0.505882,Aspen
1,2590,56,-6,390,220,151,6225,1,29,0.829038,...,212.084889,202.000000,44.799554,5180,1294.999353,5835,0.062651,0.034056,0.543590,Aspen
2,2804,139,65,3180,234,135,6121,1,12,0.656059,...,275.769832,202.333333,58.346665,25236,311.555521,2941,0.519523,0.043784,0.084277,Lodgepole Pine
3,2785,155,118,3090,238,122,6211,1,30,0.422618,...,269.235956,199.333333,66.972631,50130,154.722214,3121,0.497504,0.038963,0.078317,Lodgepole Pine
4,2595,45,-1,391,220,150,6172,1,29,0.707107,...,153.003268,201.333333,45.003704,5190,1297.499351,5781,0.063351,0.024789,0.391304,Aspen


## Remove optional drop/id columns and split X/y

In [6]:
df_model = prepare_modeling_dataframe(
    df=df,
    target_col=target_col,
    drop_cols=drop_cols,
    id_cols=id_cols,
)

X, y = split_features_target(df_model, target_col=target_col)

print("Modeling dataframe shape:", df_model.shape)
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

Modeling dataframe shape: (145890, 21)
Feature matrix shape: (145890, 20)
Target shape: (145890,)


## Identify numeric and categorical columns

In [7]:
numeric_cols, categorical_cols = split_feature_types(
    df=df_model,
    target_col=target_col,
)

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))
print("Sample numeric columns:", numeric_cols[:10])
print("Sample categorical columns:", categorical_cols[:10])

Numeric columns: 20
Categorical columns: 0
Sample numeric columns: ['elevation', 'aspect', 'vertical_distance_to_hydrology', 'horizontal_distance_to_roadways', 'hillshade_9am', 'hillshade_3pm', 'horizontal_distance_to_fire_points', 'wilderness_area', 'soil_type', 'aspect_sin']
Sample categorical columns: []


## Train/test split

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=test_size,
    random_state=random_state,
    stratify=y if use_stratify else None,
    shuffle=use_shuffle,
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (116712, 20)
X_test : (29178, 20)
y_train: (116712,)
y_test : (29178,)


# Define Models List (Code)

Logistic Regression

KNN

Decision Tree

Random Forest

XGBoost (optional)

In [9]:
models = get_baseline_models(random_state=random_state)

print("Available baseline models:")
list(models.keys())

Available baseline models:


['logistic_regression',
 'decision_tree',
 'random_forest',
 'extra_trees',
 'gradient_boosting',
 'knn',
 'gaussian_nb',
 'svc_rbf']

In [20]:
baseline_results = []
best_model_name = None
best_pipeline = None
best_evaluation = None
best_f1_macro = -1.0

for model_name, model in models.items():
    print(f"Training: {model_name}")

    pipeline = build_model_pipeline(
        numeric_cols=numeric_cols,
        categorical_cols=categorical_cols,
        model=model,
    )

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    evaluation = evaluate_classification(y_test, y_pred)
    metrics = evaluation["metrics"]

    baseline_results.append(
        {
            "model_name": model_name,
            "accuracy": metrics["accuracy"],
            "f1_macro": metrics["f1_macro"],
            "f1_weighted": metrics["f1_weighted"],
            "n_train_rows": X_train.shape[0],
            "n_test_rows": X_test.shape[0],
            "n_input_features": X_train.shape[1],
        }
    )

    print(
        f"accuracy={metrics['accuracy']:.4f} | "
        f"f1_macro={metrics['f1_macro']:.4f} | "
        f"f1_weighted={metrics['f1_weighted']:.4f}"
    )

    if metrics["f1_macro"] > best_f1_macro:
        best_f1_macro = metrics["f1_macro"]
        best_model_name = model_name
        best_pipeline = pipeline
        best_evaluation = evaluation

Training: logistic_regression


F:\DATA SCIENCE\Projects\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


<class 'dict'>
{'Aspen': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 614.0}, 'Cottonwood/Willow': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 432.0}, 'Douglas-fir': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 432.0}, 'Krummholz': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 432.0}, 'Lodgepole Pine': {'precision': 0.7206678383128295, 'recall': 0.9946153099835063, 'f1-score': 0.8357655307353661, 'support': 20614.0}, 'Ponderosa Pine': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 432.0}, 'Spruce/Fir': {'precision': 0.04132231404958678, 'recall': 0.0048216007714561235, 'f1-score': 0.008635578583765112, 'support': 6222.0}, 'accuracy': 0.7037151278360408, 'macro avg': {'precision': 0.10885573605177376, 'recall': 0.14277670153642322, 'f1-score': 0.12062872990273302, 'support': 29178.0}, 'weighted avg': {'precision': 0.5179571683116456, 'recall': 0.7037151278360408, 'f1-score': 0.5923024614616158, 'support':

# Define CV Strategy (Code)

StratifiedKFold with seed

Metric: macro F1 (and accuracy secondary)

In [10]:
cv = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=random_state,
)

scoring = {
    "f1_macro": "f1_macro",
    "accuracy": "accuracy",
}

cv

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

# Train + CV Loop (Code)

For each model:

build pipeline

run cross_val_score / cross_validate

store mean/std

In [11]:
baseline_results = []

for model_name, model in models.items():
    print(f"Running CV for: {model_name}")

    pipeline = build_model_pipeline(
        numeric_cols=numeric_cols,
        categorical_cols=categorical_cols,
        model=model,
    )

    cv_results = cross_validate(
        estimator=pipeline,
        X=X,
        y=y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
    )

    baseline_results.append(
        {
            "model_name": model_name,
            "cv_f1_macro_mean": cv_results["test_f1_macro"].mean(),
            "cv_f1_macro_std": cv_results["test_f1_macro"].std(),
            "cv_accuracy_mean": cv_results["test_accuracy"].mean(),
            "cv_accuracy_std": cv_results["test_accuracy"].std(),
            "n_features": X.shape[1],
            "n_rows": X.shape[0],
            "n_folds": n_splits,
        }
    )

Running CV for: logistic_regression
Running CV for: decision_tree
Running CV for: random_forest
Running CV for: extra_trees
Running CV for: gradient_boosting
Running CV for: knn
Running CV for: gaussian_nb
Running CV for: svc_rbf


# Results Table (Code)

Create dataframe of results

Sort by macro F1

Display

In [12]:
baseline_scores_df = pd.DataFrame(baseline_results).sort_values(
    by=["cv_f1_macro_mean", "cv_accuracy_mean"],
    ascending=False,
    ignore_index=True,
)

baseline_scores_df

,model_name,cv_f1_macro_mean,cv_f1_macro_std,cv_accuracy_mean,cv_accuracy_std,n_features,n_rows,n_folds
0,random_forest,0.911450,0.004864,0.959613,0.000949,20,145890,5
1,extra_trees,0.911337,0.004756,0.954404,0.000572,20,145890,5
2,decision_tree,0.855887,0.005020,0.935774,0.001601,20,145890,5
3,gradient_boosting,0.795543,0.005720,0.869045,0.001017,20,145890,5
4,knn,0.603832,0.007471,0.866468,0.001264,20,145890,5
5,logistic_regression,0.119753,0.000888,0.704353,0.003143,20,145890,5
6,svc_rbf,0.118470,0.000064,0.706621,0.000045,20,145890,5
7,gaussian_nb,0.005150,0.000292,0.015251,0.000078,20,145890,5


# Save Baseline Results (Code)

Save to reports/model_results/baseline_scores.csv

In [19]:
saved_baseline_scores_path = save_baseline_results(
    results=baseline_results,
    output_path=baseline_scores_file,
)

print("Baseline scores saved to:", saved_baseline_scores_path)

Baseline scores saved to: F:\DATA SCIENCE\Projects\Forest Cover Type Prediction\reports\baseline_scores.csv


# Notes (Markdown)

Which 2 models go to tuning and why

## Notes

The top 2 models to move forward to tuning should be selected based on:

- highest mean cross-validated macro F1
- stable performance (lower standard deviation across folds)
- reasonable accuracy support
- practical training cost

### Suggested decision format
- **Model 1:** Chosen because it achieved the highest macro F1 with stable fold-wise performance.
- **Model 2:** Chosen because it was close to the top model and offers a different modeling bias, which is useful for tuning comparison.

After running the notebook, replace this section with your actual model names and reasoning.